In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import scipy.io as spio
from scipy.ndimage import gaussian_filter
from scipy.ndimage.measurements import label
from scipy.spatial.distance import euclidean

def getSingleSessionPixelScaling(rawDict):
    xMax = 0
    xMin = 200
    yMax = 0
    yMin = 200
    for tr in range(len(rawDict['xPos'])):
        trxMax = np.amax(abs((rawDict['xPos'])[tr]))
        trxMin = np.amin(abs((rawDict['xPos'])[tr]))
        tryMax = np.amax(abs((rawDict['yPos'])[tr]))
        tryMin = np.amin(abs((rawDict['yPos'])[tr]))
        if trxMax > xMax:
            xMax = trxMax
        if trxMin < xMin:
            xMin = trxMin
        if tryMax > yMax:
            yMax = tryMax
        if tryMin < yMin:
            yMin = tryMin
    xScaling = 127/(xMax-xMin) #127 is from the size of the maze (0 start point)
    yScaling = 127/(yMax-yMin)
    return xScaling,yScaling,xMin,yMin

def rescalePositionValuesToCm(inputX,inputY):
    xScale,yScale,minX,minY = getSingleSessionPixelScaling({'xPos':inputX,'yPos':inputY})
    currPosX = inputX
    currPosY = inputY
    updNewPosX = [None]
    updNewPosY = [None]
    for i in range(np.shape(currPosX)[0]):
        trialPosX = np.multiply(abs(currPosX[i]),xScale)
        trialPosY = np.multiply(abs(currPosY[i]),yScale)
        updNewPosX.append(trialPosX)
        updNewPosY.append(trialPosY)
    return updNewPosX[1:],updNewPosY[1:]

def refitPositionToBoundaries(inX,inY):
    xScale,yScale,minX,minY = getSingleSessionPixelScaling({'xPos':inX,'yPos':inY})
    currPosX = inX
    currPosY = inY
    updNewPosX = [None]
    updNewPosY = [None]
    for i in range(len(currPosX)):
        trialPosX = np.multiply(abs(currPosX[i])-minX,xScale)
        trialPosY = np.multiply(abs(currPosY[i])-minY,yScale)
        updNewPosX.append(trialPosX.astype(int))
        updNewPosY.append(trialPosY.astype(int))
    return updNewPosX[1:],updNewPosY[1:]

def walkStartPosIndices(inputStartPosInd):
    outputInds = []
    for ind in inputStartPosInd:
        outputInds.append(int(ind[0][0]))
    return np.asarray(outputInds).flatten()

def getCueTrials(inputBehData):
    outCueInds = []
    for tr,arr in enumerate(inputBehData):
        if 'Cue' in arr[1]:
            outCueInds.append(tr)
    return outCueInds

def makeIntoNeededBlock(inputArray,refInds,keepBool):
    outputList = [None]
    if keepBool == False:
        for ref,singleArr in enumerate(inputArray):
            if ref not in refInds:
                outputList.append(np.asarray(singleArr).flatten())
    else:
        for ref,singleArr in enumerate(inputArray):
            if ref in refInds:
                outputList.append(np.asarray(singleArr).flatten())
    outputList = outputList[1:]
    return outputList[:5]

def loadValuesFromPosFile(inPosFile):
    pmat = spio.loadmat(inPosFile)
    regionIndex = pmat.get('regionIndex').flatten()
    cueTrialInds = getCueTrials(pmat.get('behData'))
    startPosInd = walkStartPosIndices(pmat.get('startPosInd')[0])
    xScaled,yScaled = rescalePositionValuesToCm(pmat.get('xPos')[0],pmat.get('yPos')[0])
    xPos,yPos = refitPositionToBoundaries(xScaled,yScaled)
    xPosSpa = makeIntoNeededBlock(xPos,cueTrialInds,False)
    yPosSpa = makeIntoNeededBlock(yPos,cueTrialInds,False)
    trialTimeSpa = makeIntoNeededBlock(pmat.get('trialTime')[0],cueTrialInds,False)
    spIndSpa = np.delete(startPosInd,cueTrialInds,axis=0)[:5]
    xPosCue = makeIntoNeededBlock(xPos,cueTrialInds,True)
    yPosCue = makeIntoNeededBlock(yPos,cueTrialInds,True)
    trialTimeCue = makeIntoNeededBlock(pmat.get('trialTime')[0],cueTrialInds,True)
    spIndCue = np.delete(startPosInd,cueTrialInds,axis=0)[:5]
    if len(np.shape(pmat.get('spikeTimes'))) == 3:
        spaSpikes = np.delete(pmat.get('spikeTimes'),cueTrialInds,axis=0)[:5]
        cueSpikes = np.take(pmat.get('spikeTimes'),cueTrialInds,axis=0)[:5]
    else:
        spaSpikes = np.delete(pmat.get('spikeTimes')[0],cueTrialInds,axis=0)[:5]
        cueSpikes = np.take(pmat.get('spikeTimes')[0],cueTrialInds,axis=0)[:5]
    return xPosSpa,yPosSpa,trialTimeSpa,spIndSpa,spaSpikes,xPosCue,yPosCue,trialTimeCue,spIndCue,cueSpikes

def convertGammaVectorBlockToPos(unitFields,vectorBlock):
    positionsX = []
    positionsY = []
    for ts in range(np.shape(vectorBlock)[1]):
        currSpikeVector = vectorBlock[:,ts]
        arrSum = np.zeros((128,128))
        for un,el in enumerate(currSpikeVector):
            arrSum += el * unitFields[un]
        maxX,maxY = np.unravel_index(arrSum.argmax(),arrSum.shape)
        positionsX.append(maxX)
        positionsY.append(maxY)
    return positionsX,positionsY

def buildGammaVector(inputSpikeRef,inputTimes):
    currSpikes = np.nan_to_num(inputSpikeRef,nan=0,posinf=0,neginf=0)
    gammaVectors = 0
    for i in range(len(inputTimes)):
        if i == len(inputTimes) - 1:
            lowTime = inputTimes[i]
            boolSpikeTimes = np.where(currSpikes>=lowTime,1,0)
            sumSpikeCount = np.sum(boolSpikeTimes,axis=1)/.033*np.random.uniform(0.9,1.1)
            sumSpikeCountRes = np.reshape(sumSpikeCount,(len(sumSpikeCount),1))
            gammaVectors = np.concatenate((gammaVectors,sumSpikeCountRes),axis=1)
        else:
            lowTime = inputTimes[i]
            higTime = inputTimes[int(i+1)]
            boolSpikeTimes = np.where(np.logical_and(currSpikes>=lowTime,currSpikes<=higTime),1,0)
            sumSpikeCount = np.sum(boolSpikeTimes,axis=1)/.033
            if i == 0:
                gammaVectors = np.reshape(sumSpikeCount,(len(sumSpikeCount),1))
            else:
                sumSpikeCountRes = np.reshape(sumSpikeCount,(len(sumSpikeCount),1))
                gammaVectors = np.concatenate((gammaVectors,sumSpikeCountRes),axis=1)
    return gammaVectors                

def generateEstimatedPositions(inputSimmedFile,posX,posY,time,startPos,spikeRef):
    smat = spio.loadmat(inputSimmedFile)
    allPosError = []
    trueX = [None]
    trueY = [None]
    estimX = [None]
    estimY = [None]
    fields = smat.get('unitFields')
    spikeBlock = [smat.get('Trial01'),smat.get('Trial02'),smat.get('Trial03'),smat.get('Trial04'),smat.get('Trial05')]
    for tr,arr in enumerate(time):
        currPosX = posX[tr][startPos[tr]:]
        currPosY = posY[tr][startPos[tr]:]
        singleTimeArr = arr.flatten()[startPos[tr]:]
        singleTimeArr = (singleTimeArr - singleTimeArr[0])/100000
        keepIndLength = int(len(singleTimeArr)*6.6)
        newSpikeBlock = spikeBlock[tr][:,-keepIndLength:]
        newInds = np.arange(0,np.shape(newSpikeBlock)[1],6.6).astype(int)
        gammaVectorBlock = 0
        for i in range(len(newInds)):
            if i == len(newInds) - 1:
                lowInd = newInds[i]
                singleVector = np.sum(newSpikeBlock[:,lowInd:],axis=1)
                singleVectorRes = np.reshape(singleVector,(len(singleVector),1))
            else:
                lowInd = newInds[i]
                higInd = newInds[int(i+1)]
                singleVector = np.sum(newSpikeBlock[:,lowInd:higInd],axis=1)
                singleVectorRes = np.reshape(singleVector,(len(singleVector),1))
            if i == 0:
                gammaVectorBlock = singleVectorRes
            else:
                gammaVectorBlock = np.concatenate((gammaVectorBlock,singleVectorRes),axis=1)
        trueSpikeBlock = buildGammaVector(spikeRef[tr],singleTimeArr)
        simmX,simmY = convertGammaVectorBlockToPos(fields,gammaVectorBlock)
        posError = np.sqrt((np.sum(currPosX-simmX))**2+(np.sum(currPosY-simmY))**2)/len(simmX)
        allPosError.append(posError)
        trueX.append(currPosX)
        trueY.append(currPosY)
        estimX.append(simmX)
        estimY.append(simmY)
    outDict = {'posError':allPosError,'trueX':trueX[1:],'trueY':trueY[1:],'estX':estimX[1:],'estY':estimY[1:]}
    return outDict

def pathWalk(rawPosP,rawSpaP,rawCueP,savSpaP,savCueP):
    for aRoot,aDirs,aFiles in os.walk(rawPosP):
        for aFile in aFiles:
            if aFile.endswith('.mat'):
                currRawPosFile = os.path.join(rawPosP,aFile)
                currRawSpaFile = os.path.join(rawSpaP,aFile)
                currRawCueFile = os.path.join(rawCueP,aFile)
                currSavSpaFile = os.path.join(savSpaP,aFile)
                currSavCueFile = os.path.join(savCueP,aFile)
                v1,v2,v3,v4,v5,v6,v7,v8,v9,v10 = loadValuesFromPosFile(currRawPosFile)
                try:
                    dictToSaveSpa = generateEstimatedPositions(currRawSpaFile,v1,v2,v3,v4,v5)
                    saveVar = spio.savemat(currSavSpaFile,dictToSaveSpa)
                except:
                    yes = 0
                try:
                    dictToSaveCue = generateEstimatedPositions(currRawCueFile,v6,v7,v8,v9,v10)
                    saveVar = spio.savemat(currSavCueFile,dictToSaveCue)
                except:
                    yes = 0
                print(aFile)
    return 'Done'

warnings.filterwarnings('ignore')
posRawP = ''
spaRawP = ''
cueRawP = ''
spaSavP = ''
cueSavP = ''
doneVar = pathWalk(posRawP,spaRawP,cueRawP,spaSavP,cueSavP)
print('Done')
